In [18]:
# Imports
import pandas as pd
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv
import os
import logging
import yaml
import click
import gzip, csv
#from io import StringIO

In [4]:
# Check for existence of credentials, if not found, fail early
load_dotenv(override=True)

def require_env(key: str) -> str:
    val = os.getenv(key)
    if not val:
        raise EnvironmentError(f"Umgebungsvariable {key} fehlt")
    return val

In [5]:
# Database credentials
host = require_env("HOST")
dbname = require_env("DATABASE")
user = require_env("USER")
password = require_env("PASSWORD")

DSN = f"host={host} dbname={dbname} user={user} password={password}"

In [6]:
# Read metadata: tables and column for creating datawarehouse
def read_data_contract(file_path="data_ingestion_contract.yaml"):
    try:
        # Open and parse YAML
        with open(file_path, 'r', encoding='utf-8') as f:
            yaml_data = yaml.safe_load(f)

        # Ensure YAML data is list/dict for DataFrame conversion
        if isinstance(yaml_data, dict):
            # Convert dict to list of dicts for DataFrame
            yaml_data = [yaml_data]
        elif not isinstance(yaml_data, list):
            raise ValueError("YAML content must be a list or dict to convert to DataFrame.")

        return yaml_data
    except Exception as e:
        print(f"Unexpected error: {e}")


In [7]:
data_contract = read_data_contract()
print(data_contract)

[{'listings': ['estimated_occupancy_l365d', 'number_of_reviews_ly', 'hosts_time_as_user_months', 'property_type', 'host_name', 'listing_url', 'room_type', 'number_of_reviews', 'calendar_last_scraped', 'availability_60', 'host_since', 'accommodates', 'first_review', 'maximum_nights', 'amenities', 'description', 'neighborhood_overview', 'bathrooms', 'last_review', 'review_scores_cleanliness', 'availability_365', 'minimum_nights', 'review_scores_rating', 'latitude', 'calculated_host_listings_count_shared_rooms', 'maximum_minimum_nights', 'availability_90', 'availability_30', 'price', 'number_of_reviews_l30d', 'host_profile_url', 'bathrooms_text', 'minimum_nights_avg_ntm', 'host_thumbnail_url', 'has_availability', 'name', 'price_quote_checkin_date', 'review_scores_communication', 'host_location', 'calculated_host_listings_count', 'host_listings_count', 'host_identity_verified', 'hosts_time_as_host_months', 'source', 'host_id', 'neighbourhood', 'calculated_host_listings_count_private_rooms'

In [ ]:
# Create tables and a schema for the raw data
def create_raw_schema_tables(DSN=DSN):
    with psycopg2.connect(DSN) as conn:
        with conn.cursor() as cur:

            cur.execute("CREATE SCHEMA IF NOT EXISTS airbnb_raw_data AUTHORIZATION airbnb_berlin")

            for table, columns in data_contract.items():

                column_def = sql.SQL(", ").join(sql.SQL("{} TEXT").format(
                    sql.Identifier(column)
                ) for column in columns)

                query = sql.SQL("CREATE TABLE IF NOT EXISTS {} ({})"
                                ).format(sql.Identifier(table), column_def)

                cur.execute(query)


In [ ]:
# List existing data for ingestion
def find_existing_data(filepath: str = require_env("RAWDATAPATH"))-> dict:
    candidates = {}
    for dir in os.listdir(filepath):
        if dir.startswith("data_") and os.path.isdir(dir):
            candidates[dir] = []
            for file in os.listdir(filepath + "/" + dir):
                if file.startswith(("listings","reviews")) and file.endswith(".csv.gz"):
                    candidates[dir].append(file)

    return candidates
    

In [11]:
x = find_existing_data()
print(x)

{'data_26.06.26': ['listings.csv.gz', 'reviews.csv.gz'], 'data_25.09.23': ['listings.csv.gz', 'reviews.csv.gz'], 'data_25.12.27': ['listings.csv.gz', 'reviews.csv.gz'], 'data_26.03.28': ['listings.csv.gz', 'reviews.csv.gz']}


In [34]:
y = list(find_existing_data().keys())
print(y)

['data_26.06.26', 'data_25.09.23', 'data_25.12.27', 'data_26.03.28']


In [ ]:
# Logik noch anpassen, dass beim übergeben nur einzelner Tables auch nur einzelne
# geprüft werden bei Schema drift detection und Ingestion

In [ ]:
# Detect schema drift
def read_header(folders_specified: tuple = ('data_26.06.26',)) -> list[str]:
    existing_data = find_existing_data()
    for folder in path_dict:
        for file in path_dict[folder]:
            with gzip.open(f"{folder}/{file}", "rt", encoding="utf-8", newline="") as f:
                return next(csv.reader(f))

def check_contract(path_dict: dict=find_existing_data()):
    data_contract = read_data_contract()
    for table in data_contract:
        
    unknown = data_contract 
read_header()
#data_contract = read_data_contract()


['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_profile_id',
 'host_profile_url',
 'host_name',
 'host_since',
 'hosts_time_as_user_years',
 'hosts_time_as_user_months',
 'hosts_time_as_host_years',
 'hosts_time_as_host_months',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'price_quote_checkin_date',
 'price_quote_checkout_date',
 'price_quote_total_price',
 'price_quote

In [ ]:
# Load files into the tables
def load_file(x):
    pass

In [48]:
from psycopg2 import sql

def load_file(conn, table: str, path: Path, snapshot: date, header: list[str]) -> int:
    cols = sql.SQL(", ").join(map(sql.Identifier, header))
    col_defs = sql.SQL(", ").join(sql.SQL("{} text").format(sql.Identifier(c)) for c in header)
    with conn.cursor() as cur:
        cur.execute(sql.SQL("CREATE TEMP TABLE tmp_load ({}) ON COMMIT DROP").format(col_defs))
        with gzip.open(path, "rt", encoding="utf-8", newline="") as f:
            cur.copy_expert("COPY tmp_load FROM STDIN WITH (FORMAT csv, HEADER true)", f)
        cur.execute(sql.SQL("DELETE FROM raw.{} WHERE _snapshot_date = %s").format(sql.Identifier(table)), (snapshot,))
        cur.execute(
            sql.SQL("INSERT INTO raw.{t} (_snapshot_date, _source_file, {c}) SELECT %s, %s, {c} FROM tmp_load").format(t=sql.Identifier(table), c=cols),
            (snapshot, path.name),
        )
        return cur.rowcount

NameError: name 'Path' is not defined